# 03 — Ranking by Seed

Production ranking on the committed fixture. `build_similarity` turns every
DayHike into a row of facts once and stores them, with the Calibration, next to
the Catalog. A query compares the Seed with every DayHike on each Measure the
Hiker weighs (0-100, 0 leaves it out), calibrates each Measure score against
random DayHike pairs and takes the weighted mean (ADR-0013). The Seed itself is
omitted. Several Seeds: each candidate's score is its similarity to its
**best** Seed, not the mean.

**Here:** FeatureVectors and the facts built from them, one Seed, Measure
weights, two dissimilar Seeds (max-not-mean), unknown Seed failure, DriveTime
as a stubbed query-time filter, Ranking with no month and with July, LandCover
moving a forested neighbour ahead of a barer one.

**Not here:** ingest mechanics (`01_raw_data_and_ingestion.ipynb`), Catalog
maps (`02_catalog.ipynb`), trying a new Measure (`04_rank_feature_lab.ipynb`),
the Measures on the real region (`05_similarity.ipynb`).

This notebook does not download Geofabrik extracts and does not call a live
Valhalla instance.


In [ ]:
import tempfile
from pathlib import Path

repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    fixture = candidate / "tests" / "fixtures" / "catalog.osm"
    if fixture.is_file():
        repo = candidate
        break
else:
    raise FileNotFoundError("tests/fixtures/catalog.osm")

from dayhike import build_catalog

catalog_path = Path(tempfile.mkdtemp()) / "catalog.sqlite"
catalog = build_catalog(fixture, catalog_path)
print(catalog.ingest_report)
print("sqlite", catalog.path)
print("attribution", catalog.attribution)
for hike in catalog.dayhikes:
    print(hike.name)
    print(hike.feature_vector)


## LandCover and ClimateNormal on the fixture

No WorldCover or CHELSA download. A tiny synthetic raster covers the
fixture vertices; `sample_landcover` and `sample_climate` write classes and
twelve hike-level monthly means. `build_similarity` then writes the facts and the
Calibration (`{catalog}.similarity/`); Ranking below uses them.


In [ ]:
import sqlite3

import numpy as np

from dayhike import (
    Catalog,
    build_similarity,
    geometry_store_path_for,
    sample_climate,
    sample_landcover,
)


def _write_constant_tif(path, west, south, east, north, value, dtype, nodata):
    import rasterio
    from rasterio.transform import from_bounds

    height = width = 8
    transform = from_bounds(west, south, east, north, width, height)
    with rasterio.open(
        path,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=dtype,
        crs="EPSG:4326",
        transform=transform,
        nodata=nodata,
    ) as dataset:
        dataset.write(np.full((height, width), value, dtype=dtype), 1)


store = sqlite3.connect(geometry_store_path_for(catalog_path))
try:
    west, south, east, north = store.execute(
        "SELECT MIN(longitude), MIN(latitude), MAX(longitude), MAX(latitude) FROM vertices"
    ).fetchone()
finally:
    store.close()
pad = 0.05
west, south, east, north = west - pad, south - pad, east + pad, north + pad

landcover_dir = Path(tempfile.mkdtemp())
climate_dir = Path(tempfile.mkdtemp())
_write_constant_tif(landcover_dir / "cover.tif", west, south, east, north, 10, "uint8", 0)
for month in range(1, 13):
    tas = 21.0 if month == 7 else 0.0
    _write_constant_tif(
        climate_dir / f"chelsa_tas_{month:02d}.tif",
        west,
        south,
        east,
        north,
        tas,
        "float32",
        -9999.0,
    )
    _write_constant_tif(
        climate_dir / f"chelsa_pr_{month:02d}.tif",
        west,
        south,
        east,
        north,
        40.0,
        "float32",
        -9999.0,
    )
    _write_constant_tif(
        climate_dir / f"chelsa_rsds_{month:02d}.tif",
        west,
        south,
        east,
        north,
        12000.0,
        "float32",
        -9999.0,
    )
sample_landcover(catalog_path, landcover_dir)
sample_climate(catalog_path, climate_dir)
print("similarity", build_similarity(catalog_path, Path(tempfile.mkdtemp())))
catalog = Catalog.open(catalog_path)


## Facts and Measures

The Catalog FeatureVector (printed above) plus the DayHikeGeometry give each
DayHike one column of facts. Each Measure compares one fact of two DayHikes and
gives 1 (alike) to 0 (unlike), or nothing when either lacks the fact. Missing
elevation, `sac_scale`, or unsampled rasters stay missing (NaN), never 0.


In [ ]:
import os

if "PYTEST_CURRENT_TEST" in os.environ:
    import matplotlib as mpl

    mpl.use("Agg")

import matplotlib.pyplot as plt


def show_fig(fig):
    try:
        from IPython.display import display

        display(fig)
    except ImportError:
        pass
    plt.close(fig)
import numpy as np
import pandas as pd

from dayhike import MEASURE_NAMES
from dayhike.similarity import FACT_COLUMNS, MEASURES

store = catalog.similarity
facts = pd.DataFrame(
    np.asarray(store.facts).T,
    columns=FACT_COLUMNS,
    index=[hike.name for hike in catalog.dayhikes],
)
shown = [name for name in FACT_COLUMNS if not name[-3:-2] == "_"]  # no month columns
print(facts[shown].T.to_string())
print("measures", len(MEASURE_NAMES))
for measure in MEASURES:
    print(f"  {measure.name}\t{measure.kind}\t{measure.constant}")

fig, ax = plt.subplots(figsize=(9, 3.5))
values = facts[shown].to_numpy(dtype=float)
filled = np.where(np.isnan(values), 0.0, values)
# column-wise min-max so mixed units are visible together
span = filled.max(axis=0) - filled.min(axis=0)
span[span == 0] = 1.0
normed = (filled - filled.min(axis=0)) / span
im = ax.imshow(normed, aspect="auto", cmap="YlGn")
ax.set_yticks(range(len(facts)))
ax.set_yticklabels(facts.index)
ax.set_xticks(range(len(shown)))
ax.set_xticklabels(shown, rotation=70, ha="right", fontsize=7)
ax.set_title("Similarity facts (min-max per column, NaN→0)")
fig.colorbar(im, ax=ax, fraction=0.03)
fig.tight_layout()
show_fig(fig)


## Ranking by Seed

One Seed omits itself and ranks the rest. With no weights the core Measures
count 100 and the others are off (ADR-0015). `score` is the similarity, the
weighted mean of the calibrated Measure scores, so about 0.5 is a typical
random pair.

The fixture has only four DayHikes, so `variant` copies the Catalog, renames and
reshapes two of them with SQL and rebuilds the facts. Two dissimilar Seeds score
each candidate by its **best** Seed, not the mean — otherwise a ridge and a
valley would collapse into a mushy middle. `score_rows` gives the same
similarity for any rows.


In [ ]:
import shutil

from dayhike import rank_dayhikes
from dayhike.similarity import score_rows

for item in rank_dayhikes(catalog, ["Ridge Walk"]).items:
    print(f"one_seed\t{item.dayhike.name}\t{item.score}")

for item in rank_dayhikes(catalog, ["Valley Path"]).items:
    print(f"one_seed_valley\t{item.dayhike.name}\t{item.score}")


def variant(*statements):
    """A copy of this Catalog changed by SQL statements, with its facts rebuilt."""
    path = Path(tempfile.mkdtemp()) / "catalog.sqlite"
    shutil.copy(catalog_path, path)
    shutil.copy(geometry_store_path_for(catalog_path), geometry_store_path_for(path))
    connection = sqlite3.connect(path)
    for statement in statements:
        connection.execute(statement)
    connection.commit()
    connection.close()
    build_similarity(path, path.parent / "dem")
    return Catalog.open(path)


# relation:1003 becomes a copy of Ridge Walk's size; PS-1 sits between the two.
extended = variant(
    "UPDATE dayhikes SET name = 'Near Ridge' WHERE id = 1003",
    "UPDATE dayhikes SET name = 'Mid Path' WHERE id = 1007",
    "UPDATE feature_vectors SET (length_m, elevation_min_m, elevation_max_m, difficulty_ordinal) = "
    "(SELECT length_m, elevation_min_m, elevation_max_m, difficulty_ordinal FROM feature_vectors "
    "WHERE dayhike_id = 1001) WHERE dayhike_id = 1003",
    "UPDATE feature_vectors SET (length_m, elevation_min_m, elevation_max_m, difficulty_ordinal) = "
    "(SELECT 7500, elevation_min_m, elevation_max_m, 1 FROM feature_vectors "
    "WHERE dayhike_id = 1001) WHERE dayhike_id = 1007",
)
SIZE = {"length": 100, "highest": 100, "lowest": 100, "difficulty": 100}
for item in rank_dayhikes(extended, ["Ridge Walk"], SIZE).items:
    print(f"size_weights\t{item.dayhike.name}\t{item.score}")
both = rank_dayhikes(extended, ["Ridge Walk", "Valley Path"], SIZE)
for item in both.items:
    print(f"two_seeds\t{item.dayhike.name}\t{item.score}")
rows = np.array([2, 3])  # Near Ridge and Mid Path, in id order
only_ridge, _ = score_rows(extended.similarity, [0], rows, SIZE)
only_valley, _ = score_rows(extended.similarity, [1], rows, SIZE)
two, _ = score_rows(extended.similarity, [0, 1], rows, SIZE)
for name, score, ridge_score, valley_score in zip(
    ("Near Ridge", "Mid Path"), two, only_ridge, only_valley
):
    expected_max = np.nanmax([ridge_score, valley_score])
    expected_mean = np.nanmean([ridge_score, valley_score])
    print(f"max_not_mean\t{name}\t{score}\t{expected_max}\t{expected_mean}")


In [ ]:
ridge_rank = rank_dayhikes(catalog, ["Ridge Walk"])
fig, ax = plt.subplots(figsize=(7, 3.5))
names = [item.dayhike.name for item in ridge_rank.items]
scores = [item.score for item in ridge_rank.items]
ax.barh(names[::-1], scores[::-1], color="#1d4ed8")
ax.set_xlabel("score (weighted mean of calibrated Measures)")
ax.set_title("Neighbours of Ridge Walk")
fig.tight_layout()
show_fig(fig)


## QueryMonth and LandCover

No month uses the June–August regime temperature. Month 7 uses July. Those
two temperatures are not the same on this synthetic raster (July is a spike).
The synthetic LandCover raster makes every fixture DayHike forest, so a variant
turns Valley Path half bare and PS-1 into a bare copy: weighed on LandCover, a
forested Seed ranks the forested neighbour ahead of the barer ones.


In [ ]:
from dayhike import rank_dayhikes

ridge = next(hike for hike in catalog.dayhikes if hike.name == "Ridge Walk")
july = ridge.climate_months[6]
print(f"july_temperature\t{july[0]}")
print(f"regime_temperature\t{ridge.feature_vector.temperature_c}")

for item in rank_dayhikes(catalog, ["Ridge Walk"]).items:
    print(f"no_month\t{item.dayhike.name}\t{item.score}")
for item in rank_dayhikes(catalog, ["Ridge Walk"], month=7).items:
    print(f"july\t{item.dayhike.name}\t{item.score}")

cover_catalog = variant(
    "UPDATE dayhikes SET name = 'Forested Twin' WHERE id = 1003",
    "UPDATE dayhikes SET name = 'Bare Twin' WHERE id = 1007",
    "UPDATE feature_vectors SET tree_fraction = 0.5, bare_fraction = 0.5 WHERE dayhike_id = 1002",
    "UPDATE feature_vectors SET tree_fraction = 0.0, bare_fraction = 1.0 WHERE dayhike_id = 1007",
)
for item in rank_dayhikes(cover_catalog, ["Ridge Walk"], {"landcover": 100}).items:
    print(f"landcover_rank\t{item.dayhike.name}\t{item.score}")


## Unknown Seed

A Seed name that is not in the Catalog fails loudly. There is no fuzzy lookup.


In [ ]:
from dayhike import UnknownSeedName, rank_dayhikes

try:
    rank_dayhikes(catalog, ["Unknown Ridge"])
except UnknownSeedName as exc:
    print(type(exc).__name__)
    print(exc)


## DriveTime filter (stub client)

DriveTime is a query-time filter from today's Origin, not a fact or a
Measure. This cell stubs the Valhalla matrix client so CI never needs routing
tiles. Real routing from a live Origin needs local Valhalla tiles.

Valhalla is **not** a pip package and is **not** started by this notebook. It
is a Docker service (`data/valhalla/compose.yaml`) that listens on
`http://127.0.0.1:8002` after `prepare-roads.sh` writes a car-only PBF into
`data/valhalla/` and you run compose. `dayhike` then POSTs `/sources_to_targets`.
Ranking without Origin never connects. A drive-time pre-filter can pass the
reachable DayHike ids to `rank_dayhikes(..., allowed_ids=...)` instead.


In [ ]:
from dayhike import Origin, filter_by_drive_time, rank_dayhikes


class StubMatrix:
    def __init__(self, time_s):
        self._time_s = time_s

    def sources_to_targets(self, origin, trailheads):
        return [self._time_s] * len(trailheads)


origin = Origin(latitude=45.8, longitude=15.97)
ranking = rank_dayhikes(catalog, ["Ridge Walk"])
kept = filter_by_drive_time(ranking, origin, 2, StubMatrix(1800.0))
for item in kept.items:
    print(f"drive_keep\t{item.dayhike.name}\t{item.drive_time.hours}")
dropped = filter_by_drive_time(ranking, origin, 2, StubMatrix(None))
print(f"drive_drop_count\t{len(dropped.items)}")


## Limitations

- The Catalog unit is `route=hiking` relations in a length band (1–20 km). Unnamed `highway=path` ways never become DayHikes; neither do superroutes or out-of-band lengths. Blank OSM `name` is kept with a Seed label and `name_origin`.
- DriveTime for a real Origin needs local Valhalla tiles; the stub above is only a stand-in.
- There are no scraped reviews and no webpage ratings yet. Ratings and UI are future work.
- Ingest reads OSM XML or PBF (pyosmium). `dayhike fetch-extracts` downloads the Geofabrik list.
- AmenityFacts count nodes, ways, and tagged relations within 20 km of the Trailhead.

**Next:** [`04_rank_feature_lab.ipynb`](04_rank_feature_lab.ipynb) is the
sandbox for trying a new fact as a Measure against this same scorer.
